In [1]:
import pyodbc
import pandas as pd
import geopy.distance  # Pour calculer les distances géographiques

# Connexion à la base de données SQL Server
conn = pyodbc.connect('DRIVER={SQL Server};SERVER=LAPTOP-HED7JE2F;DATABASE=PGS_STAGE_DW;UID=sa;PWD=sa@123@123')
cursor = conn.cursor()

# Récupération des données
query = """
    SELECT
        T.TRAILER_CODE,
        O.ORDER_ID,
        O.TRIP_ID,
        C.CUSTOMER_NUMBER,
        G.GOUVERNORAT
    FROM
        DIM_ORDER_TRAILER T
        INNER JOIN DIM_ORDERS O ON T.ORDER_ID = O.ORDER_ID
        INNER JOIN DIM_CUSTOMER C ON O.CUSTOMER_NUMBER = C.CUSTOMER_NUMBER
        INNER JOIN DIM_GOUVERNERAT G ON C.BILLING_ADDRESS_1 = G.ADRESSE
"""
df = pd.read_sql(query, conn)

# Coordonnées des gouvernorats (à remplacer par les vraies coordonnées)
gouvernorat_coords = {
    'Tunis': (36.8065, 10.1815),
    'Sousse': (35.8256, 10.6084),
    'Sfax': (34.7403, 10.7618),
    'le Kef': (36.1700, 8.7071),
    'Jendouba': (36.5000, 8.7833),
    'Mahdia': (35.5046, 11.0624),
    'Médenine': (33.3585, 10.5105),
    'Ben Arous': (36.7205, 10.1853),
    'Béja': (36.7256, 9.1800),
    'Kairouan': (35.6784, 10.0984),
    'Nabeul': (36.4561, 10.7376),
    'Gabès': (33.8790, 10.0980),
    'Ariana': (36.8660, 10.1915),
    'Kasserine': (35.1667, 8.8333),
    'Tataouine': (33.4069, 10.4518),
    'Monastir': (35.7715, 10.8262),
    'Tozeur': (33.9121, 8.1335),
    'Siliana': (36.0833, 9.4000),
    'Gafsa': (34.4254, 8.7834),
    'Zaghouan': (36.4061, 10.1414),
    'Sidi Bouzid': (35.1167, 9.5000),
}

# Coordonnées du terminal de départ
terminal_coords = (36.829, 10.230)

def calculate_distance(start_coords, end_coords):
    return geopy.distance.geodesic(start_coords, end_coords).km

# Calcul des distances par trailer et par trip
trailer_distances = {}
trip_distances = {}

# Filtrer les gouvernorats NULL
df = df[df['GOUVERNORAT'].notna() & df['GOUVERNORAT'].isin(gouvernorat_coords.keys())]

for trip_id, trip_data in df.groupby('TRIP_ID'):
    # Point de départ pour le trip
    current_location = terminal_coords
    total_distance = 0
   
    # Calcul des distances pour chaque trailer dans le trip
    for _, row in trip_data.iterrows():
        gouvernorat = row['GOUVERNORAT']
        trailer_code = row['TRAILER_CODE']
       
        # Calculer la distance vers la destination
        distance = calculate_distance(current_location, gouvernorat_coords[gouvernorat])
       
        # Ajouter la distance pour ce trailer
        if trailer_code not in trailer_distances:
            trailer_distances[trailer_code] = 0
        trailer_distances[trailer_code] += distance
       
        # Mettre à jour la position actuelle
        current_location = gouvernorat_coords[gouvernorat]
       
        # Ajouter la distance à la distance totale du trip
        total_distance += distance

    # Stocker la distance totale pour ce trip
    trip_distances[trip_id] = total_distance

# Création de la nouvelle table TRAILERS_KILOMETERS
cursor.execute("""
    IF OBJECT_ID('TRAILERS_KILOMETERS', 'U') IS NOT NULL
        DROP TABLE TRAILERS_KILOMETERS;
   
    CREATE TABLE TRAILERS_KILOMETERS (
        TRAILER_CODE VARCHAR(50),
        DISTANCE_KM FLOAT
    )
""")

# Insertion des résultats dans la table TRAILERS_KILOMETERS
for trailer_code, distance in trailer_distances.items():
    cursor.execute("""
        INSERT INTO TRAILERS_KILOMETERS (TRAILER_CODE, DISTANCE_KM)
        VALUES (?, ?)
    """, trailer_code, distance)

# Sauvegarde des modifications
conn.commit()

# Fermeture de la connexion à la base de données
conn.close()

print("Les distances ont été enregistrées dans la table TRAILERS_KILOMETERS.")

C:\Users\user\AppData\Local\Temp\ipykernel_40552\2549514587.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Les distances ont été enregistrées dans la table TRAILERS_KILOMETERS.
